In [1]:
"""
해밍 부호 Hamming(7,4) 오류 정정 실습 프로그램.

개요(강의 자료의 전형적인 정의와 동일한 배치):
- 코드 길이 n=7, 정보 비트 k=4 → (7,4) 선형 블록부호.
- 비트 위치는 1부터 7까지 번호를 매기며, 위치 번호가 2의 거듭제곱(1,2,4)인 자리는 검사(parity) 비트,
  나머지(3,5,6,7)는 정보(data) 비트를 둔다.
- 짝수 패리티(각 검사 집합의 XOR=0)를 사용한다.

검사 방정식(짝수 패리티):
- p1(위치1): 위치 1,3,5,7 의 XOR = 0  (위치 번호의 LSB가 1인 것들)
- p2(위치2): 위치 2,3,6,7 의 XOR = 0
- p4(위치4): 위치 4,5,6,7 의 XOR = 0

오류 정정:
- 수신어에서 위 세 검사의 패리티 불일치 여부를 이진수로 읽으면 신드롬이 되고,
  그 값이 오류 비트의 위치(1~7)를 가리킨다. (단일 비트 오류 가정)
- 신드롬이 0이면 오류 없음으로 판단한다. (2비트 이상 오류에서는 오판 가능)

사용 방법:
1) 스크립트를 실행한다.
2) 4비트 정보열을 입력하거나, 직접 7비트 코드워드를 입력할 수 있다.
3) 전송 중 비트 반전 위치를 입력해 오류를 주입한다.
4) 신드롬 계산, 정정, 복원된 정보 비트를 확인한다.
"""

from __future__ import annotations


def validate_bits(bits: str) -> bool:
    return len(bits) > 0 and all(ch in {"0", "1"} for ch in bits)


def flip_bit(bit: str) -> str:
    return "1" if bit == "0" else "0"


def xor_bits(chars: list[str]) -> int:
    return sum(1 for ch in chars if ch == "1") % 2


def hamming74_encode(data4: str) -> str:
    if len(data4) != 4:
        raise ValueError("정보 비트는 정확히 4비트여야 합니다.")
    if not validate_bits(data4):
        raise ValueError("0/1 비트열만 허용됩니다.")

    d1, d2, d3, d4 = (int(ch) for ch in data4)
    # 위치 3,5,6,7 에 정보 비트
    p1 = d1 ^ d2 ^ d4
    p2 = d1 ^ d3 ^ d4
    p4 = d2 ^ d3 ^ d4
    # 위치: 1 p1, 2 p2, 3 d1, 4 p4, 5 d2, 6 d3, 7 d4
    parts = [p1, p2, d1, p4, d2, d3, d4]
    return "".join(str(b) for b in parts)


def hamming74_syndrome(codeword7: str) -> tuple[int, int, int, int]:
    """
    returns: (s1, s2, s4, syndrome_value)
    syndrome_value = s4*4 + s2*2 + s1 (이진수 s4s2s1)
    """
    if len(codeword7) != 7:
        raise ValueError("코드워드는 정확히 7비트여야 합니다.")
    if not validate_bits(codeword7):
        raise ValueError("0/1 비트열만 허용됩니다.")

    b = list(codeword7)  # index 0 -> position 1
    s1 = xor_bits([b[0], b[2], b[4], b[6]])  # 1,3,5,7
    s2 = xor_bits([b[1], b[2], b[5], b[6]])  # 2,3,6,7
    s4 = xor_bits([b[3], b[4], b[5], b[6]])  # 4,5,6,7
    syn = (s4 << 2) | (s2 << 1) | s1
    return s1, s2, s4, syn


def hamming74_correct(codeword7: str) -> tuple[str, int, int, str]:
    """
    returns: (corrected_codeword, syndrome, error_position_0_if_none, note)
    """
    s1, s2, s4, syn = hamming74_syndrome(codeword7)
    bits = list(codeword7)

    if syn == 0:
        return codeword7, syn, 0, "신드롬 0: 단일 비트 오류가 없다고 판단합니다."

    if syn < 1 or syn > 7:
        return codeword7, syn, 0, "비정상 신드롬(이론상 발생하지 않음)."

    idx = syn - 1
    bits[idx] = flip_bit(bits[idx])
    corrected = "".join(bits)
    note = f"신드롬 {syn}: 위치 {syn} 비트를 반전해 정정했습니다."
    return corrected, syn, syn, note


def hamming74_extract_data(codeword7: str) -> str:
    """위치 3,5,6,7 의 비트를 순서대로 이어붙인 정보열."""
    return codeword7[2] + codeword7[4] + codeword7[5] + codeword7[6]


def inject_errors(codeword: str, positions_1_based: list[int]) -> str:
    bits = list(codeword)
    for pos in positions_1_based:
        idx = pos - 1
        bits[idx] = flip_bit(bits[idx])
    return "".join(bits)


def parse_error_positions(raw: str, max_len: int) -> list[int]:
    if not raw.strip():
        return []

    tokens = raw.replace(",", " ").split()
    positions: list[int] = []
    for t in tokens:
        if not t.isdigit():
            raise ValueError(f"숫자가 아닌 입력: {t}")
        value = int(t)
        if value < 1 or value > max_len:
            raise ValueError(f"범위를 벗어난 위치: {value} (허용: 1~{max_len})")
        positions.append(value)
    return positions


def explain_checks(cw: str) -> None:
    b = list(cw)
    c1 = xor_bits([b[0], b[2], b[4], b[6]])
    c2 = xor_bits([b[1], b[2], b[5], b[6]])
    c4 = xor_bits([b[3], b[4], b[5], b[6]])
    print("\n[검사식(짝수 패리티) XOR 값]")
    print(f"- 검사1 (위치 1,3,5,7): {b[0]}+{b[2]}+{b[4]}+{b[6]} -> XOR={c1}")
    print(f"- 검사2 (위치 2,3,6,7): {b[1]}+{b[2]}+{b[5]}+{b[6]} -> XOR={c2}")
    print(f"- 검사4 (위치 4,5,6,7): {b[3]}+{b[4]}+{b[5]}+{b[6]} -> XOR={c4}")
    print(f"- 신드롬 (이진 s4s2s1): {c4}{c2}{c1} (십진 {c4 * 4 + c2 * 2 + c1})")


def run_interactive() -> None:
    print("=" * 60)
    print("해밍 부호 Hamming(7,4) 오류 정정 실습")
    print("=" * 60)

    print("\n[입력 방식]")
    print("1) 4비트 정보만 입력 → 프로그램이 패리티를 붙여 7비트 코드워드를 생성")
    print("2) 7비트 코드워드 직접 입력 → (이미 부호화된 경우) 그대로 실습")

    while True:
        bits_in = input("\n비트열 입력(4비트 또는 7비트): ").strip()
        if not validate_bits(bits_in):
            print("입력 오류: 0과 1로만 구성된 비트열을 입력하세요.")
            continue
        if len(bits_in) == 4:
            cw = hamming74_encode(bits_in)
            print("\n[송신 측]")
            print(f"- 입력 정보(위치 3,5,6,7에 배치): {bits_in}")
            print(f"- 생성 코드워드(위치 1~7):          {cw}")
            print("  (위치 1,2,4 = 검사비트 / 3,5,6,7 = 정보비트)")
            break
        if len(bits_in) == 7:
            cw = bits_in
            print("\n[송신 측]")
            print(f"- 입력 코드워드(7비트): {cw}")
            s1, s2, s4, syn0 = hamming74_syndrome(cw)
            if syn0 != 0:
                print(f"  주의: 입력 코드워드 자체의 신드롬이 {syn0}입니다(이미 오류이거나 임의 입력).")
            else:
                print("  입력 코드워드의 신드롬: 0 (검사식을 만족)")
            break
        print("입력 오류: 길이는 4 또는 7이어야 합니다.")

    print("\n[전송 중 오류 주입]")
    print("- 위치는 1부터 시작, 범위는 1~7")
    print("- 예: 3 또는 2,5 (여러 개면 다중 비트 오류; 단일 비트 정정 가정이 깨질 수 있음)")
    print("- 엔터만 누르면 오류 없이 전송")

    while True:
        raw = input("비트 반전 위치 입력: ").strip()
        try:
            error_positions = parse_error_positions(raw, 7)
            break
        except ValueError as ex:
            print(f"입력 오류: {ex}")

    received = inject_errors(cw, error_positions)

    print("\n[수신 측]")
    print(f"- 수신 코드워드: {received}")
    explain_checks(received)

    corrected, syn, err_pos, note = hamming74_correct(received)
    print(f"\n[신드롬/정정]")
    print(f"- {note}")
    print(f"- 정정 후 코드워드: {corrected}")

    data_sent = hamming74_extract_data(cw)
    data_got_raw = hamming74_extract_data(received)
    data_got_fixed = hamming74_extract_data(corrected)

    print("\n[정보 비트 복원]")
    print(f"- 송신 정보(참고):     {data_sent}")
    print(f"- 정정 전 추출 정보: {data_got_raw}")
    print(f"- 정정 후 추출 정보: {data_got_fixed}")

    if len(error_positions) == 0:
        print("\n[결론] 오류를 주입하지 않았습니다.")
    elif len(error_positions) == 1:
        if data_got_fixed == data_sent:
            print("\n[결론] 단일 비트 오류를 정정하여 원래 정보로 복원했습니다.")
        else:
            print("\n[결론] 정정 후에도 정보가 원문과 다릅니다(비정상 케이스).")
    else:
        print("\n[결론] 다중 비트 오류는 Hamming(7,4)의 단일 비트 정정 모델을 벗어납니다.")
        print("       신드롬이 0이 아닐 수 있으나, 잘못된 위치를 뒤집어 오정정될 수 있습니다.")


if __name__ == "__main__":
    run_interactive()


해밍 부호 Hamming(7,4) 오류 정정 실습

[입력 방식]
1) 4비트 정보만 입력 → 프로그램이 패리티를 붙여 7비트 코드워드를 생성
2) 7비트 코드워드 직접 입력 → (이미 부호화된 경우) 그대로 실습



비트열 입력(4비트 또는 7비트):  1110



[송신 측]
- 입력 정보(위치 3,5,6,7에 배치): 1110
- 생성 코드워드(위치 1~7):          0010110
  (위치 1,2,4 = 검사비트 / 3,5,6,7 = 정보비트)

[전송 중 오류 주입]
- 위치는 1부터 시작, 범위는 1~7
- 예: 3 또는 2,5 (여러 개면 다중 비트 오류; 단일 비트 정정 가정이 깨질 수 있음)
- 엔터만 누르면 오류 없이 전송


비트 반전 위치 입력:  3



[수신 측]
- 수신 코드워드: 0000110

[검사식(짝수 패리티) XOR 값]
- 검사1 (위치 1,3,5,7): 0+0+1+0 -> XOR=1
- 검사2 (위치 2,3,6,7): 0+0+1+0 -> XOR=1
- 검사4 (위치 4,5,6,7): 0+1+1+0 -> XOR=0
- 신드롬 (이진 s4s2s1): 011 (십진 3)

[신드롬/정정]
- 신드롬 3: 위치 3 비트를 반전해 정정했습니다.
- 정정 후 코드워드: 0010110

[정보 비트 복원]
- 송신 정보(참고):     1110
- 정정 전 추출 정보: 0110
- 정정 후 추출 정보: 1110

[결론] 단일 비트 오류를 정정하여 원래 정보로 복원했습니다.
